<a href="https://colab.research.google.com/github/jivaniaadit/factor_xa_cheminformatics/blob/main/notebook/week1_day2_descriptors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1
!pip install rdkit -q

# Cell 2 (markdown)
# Week 1, Day 2: Molecular Properties and Lipinski's Rule

# Cell 3
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors, Draw
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 39.8 MB/s eta 0:00:00


In [ ]:
mol = Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O')

print(f"MW: {Descriptors.MolWt(mol):.2f}")
print(f"LogP: {Descriptors.MolLogP(mol):.2f}")
print(f"HBD: {Lipinski.NumHDonors(mol)}")
print(f"HBA: {Lipinski.NumHAcceptors(mol)}")
print(f"TPSA: {Descriptors.TPSA(mol):.2f}")
print(f"Rotatable bonds: {Lipinski.NumRotatableBonds(mol)}")

MW: 180.16
LogP: 1.31
HBD: 1
HBA: 3
TPSA: 63.60
Rotatable bonds: 2


In [ ]:
drugs = {
    'Aspirin': 'CC(=O)Oc1ccccc1C(=O)O',
    'Ibuprofen': 'CC(C)Cc1ccc(C(C)C(=O)O)cc1',
    'Acetaminophen': 'CC(=O)Nc1ccc(O)cc1',
    'Caffeine': 'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',
    'Penicillin G': 'CC1(C)SC2C(NC(=O)Cc3ccccc3)C(=O)N2C1C(=O)O',
    'Diazepam': 'CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21',
    'Propranolol': 'CC(C)NCC(O)COc1cccc2ccccc12',
    'Fluoxetine': 'CNCCC(Oc1ccc(C(F)(F)F)cc1)c1ccccc1',
    'Atorvastatin': 'CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F)cc2)n1CC[C@@H](O)C[C@@H](O)CC(=O)O',
    'Metformin': 'CN(C)C(=N)NC(=N)N',
}

def compute_properties(mol):
    return {
        'MW': Descriptors.MolWt(mol),
        'LogP': Descriptors.MolLogP(mol),
        'HBD': rdMolDescriptors.CalcNumHBD(mol),
        'HBA': rdMolDescriptors.CalcNumHBA(mol),
        'TPSA': rdMolDescriptors.CalcTPSA(mol),
        'RotBonds': Descriptors.NumRotatableBonds(mol)
    }

results = []
for name, smiles in drugs.items():
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        continue
    props = compute_properties(mol)
    props['Name'] = name
    results.append(props)

df = pd.DataFrame(results)

df

,MW,LogP,HBD,HBA,TPSA,RotBonds,Name
0,180.159,1.31010,1,3,63.60,2,Aspirin
1,206.285,3.07320,1,1,37.30,4,Ibuprofen
2,151.165,1.35060,2,2,49.33,1,Acetaminophen
3,194.194,-1.02930,0,3,61.82,0,Caffeine
4,334.397,0.86080,2,4,86.71,4,Penicillin G
5,284.746,3.15380,0,2,32.67,1,Diazepam
6,259.349,2.57750,2,3,41.49,6,Propranolol
7,309.331,4.43500,1,2,21.26,6,Fluoxetine
8,558.650,6.31360,4,4,111.79,12,Atorvastatin
9,129.167,-1.03416,4,2,88.99,0,Metformin


In [ ]:
def lipinski_violations(row):
    violations = 0
    if row['MW'] > 500: violations += 1
    if row['LogP'] > 5: violations += 1
    if row['HBD'] > 5: violations += 1
    if row['HBA'] > 10: violations += 1
    return violations

df['LipinskiViolations'] = df.apply(lipinski_violations, axis=1)
df

,MW,LogP,HBD,HBA,TPSA,RotBonds,Name,LipinskiViolations
0,180.159,1.31010,1,3,63.60,2,Aspirin,0
1,206.285,3.07320,1,1,37.30,4,Ibuprofen,0
2,151.165,1.35060,2,2,49.33,1,Acetaminophen,0
3,194.194,-1.02930,0,3,61.82,0,Caffeine,0
4,334.397,0.86080,2,4,86.71,4,Penicillin G,0
5,284.746,3.15380,0,2,32.67,1,Diazepam,0
6,259.349,2.57750,2,3,41.49,6,Propranolol,0
7,309.331,4.43500,1,2,21.26,6,Fluoxetine,0
8,558.650,6.31360,4,4,111.79,12,Atorvastatin,2
9,129.167,-1.03416,4,2,88.99,0,Metformin,0
